# `parse_pdf` — analyse complète d'un PDF en 1 ouverture fitz

Le script `parse_pdf.py` ouvre fitz **une seule fois** et retourne 4 sorties prêtes à être consommées en aval (LLM, indexation sémantique, base de données, API) :

| Sortie | Schéma |
|---|---|
| `line_df`  | 1 ligne = 1 ligne de texte du PDF (texte, bbox, font, taille, `is_invisible` pour distinguer texte natif vs couche OCR) |
| `image_df` | 1 ligne = 1 image embarquée (bbox affichée en points PDF, dimensions intrinsèques, ratio de couverture) |
| `page_df`  | 1 ligne = 1 page (`page_type`, flags additifs, comptes, `extraction_strategy`, `tool`) |
| `doc_summary` | JSON synthèse : `source_tool`, `content_type`, `recommended_strategy`, `pages_needing_ocr`, `ocr_quality`, … |

**page_type** (mutuellement exclusif) :

| page_type | Critère |
|---|---|
| `native`              | Fonts + texte natif, pas d'image plein format |
| `native_with_image`   | Fonts + texte natif + images intégrées (logos, schémas) |
| `scanned`             | Image ≥ 85 % de la page, sans couche OCR |
| `scanned_ocr_good`    | Image plein format + couche OCR exploitable |
| `scanned_ocr_bad`     | Image plein format + couche OCR dégradée (à ré-OCR-iser) |
| `mixed`               | Image plein format ET texte natif distinct |
| `empty`               | Page quasi vide |
| `unknown`             | Aucun signal décisif |

**extraction_strategy** (routage pour le pipeline aval) : `native` (fitz direct), `ocr` (OCR obligatoire), `hybrid` (texte natif + OCR sur images), `skip` (page vide).

**Type doc global** (`content_type`) : si ≥ 95 % des pages utiles dans une seule catégorie → cette catégorie ; sinon → `mixed`.

In [ ]:
# À exécuter une seule fois : installer le package en mode editable (src layout)
import sys
!{sys.executable} -m pip install -q -e ../../../..

## 1 — Démonstration sur un PDF unique

On lance la fonction sur un seul PDF et on affiche le `doc_summary` + le `page_df`.

In [ ]:
import json
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

PDF = '../../../../../d_actif_pro_sant2.pdf'   # ← input
result = parse_pdf(PDF)

print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
print('=== page_df ===')
print(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images', 'image_coverage_ratio']].to_string(index=False))
print()
print(f"line_df  : {len(result['line_df'])} lignes")
print(f"image_df : {len(result['image_df'])} images")

## 2 — Banque de cas (test visuel sur le corpus client)

Couvre les profils typiques rencontrés sur le corpus :
- contrat MRH InDesign (`native_with_image`)
- rapport de solvabilité (creator inconnu)
- contrat avec page scannée (`mixed`)
- rapport recompressé Ghostscript

Pour chaque PDF on lit la sortie minimale qu'on attend ; à l'œil on valide que la classification est cohérente.

In [ ]:
from pathlib import Path
import pandas as pd
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

REPO_ROOT = Path('../../../..').resolve()

CASES = [
    # (chemin relatif depuis repo root, profil attendu commenté)
    ('data/CG contrats MRH/CG allianz MRH.pdf',                 'InDesign / native'),
    ('data/CG contrats MRH/CG Habitation_MMA_410s.pdf',         'InDesign / native'),
    ('data/CG contrats MRH/CG Assistance aux personnes AXA.pdf','InDesign / mixed'),
    ('data/annuel_reports/ABEILLE ASSURANCES/09 SFCR 2022 - 10 Abeille Vie.pdf', 'Unknown / native'),
    ('data/cmo/CMO-April-2024.pdf',                             'Ghostscript / native'),
    ('data/insurance/agcs euroapi.pdf',                         'Microsoft Word / native'),
    ('data/paper/1706.03762v7.pdf',                             'Unknown / native (Attention is all you need)'),
    ('data/reports/sdg_7_2025.pdf',                             'InDesign / mixed (gros doc)'),
]

rows = []
for rel, expected in CASES:
    pdf = REPO_ROOT / rel
    if not pdf.exists():
        rows.append({'file': rel, 'expected': expected, 'OK': '(missing)'})
        continue
    s = parse_pdf(pdf)['doc_summary']
    rows.append({
        'file':          Path(rel).name,
        'expected':      expected,
        'n_pages':       s['n_pages'],
        'source_tool':   s['source_tool'],
        'content_type':  s['content_type'],
        'reco':          s['recommended_strategy'],
        'pages_OCR':     len(s['pages_needing_ocr']),
    })

print(pd.DataFrame(rows).to_string(index=False))

## 3 — Bench complet

Pour le bench sur les 71 PDFs du corpus, voir `notebooks/bench_parse_pdf_js.ipynb`. Sortie consolidée : `bench_parse_pdf_results_js.csv` à la racine du repo (71 PDFs traités, 6 589 pages, 0 erreur, 43 min).